# Lab 06 — SMS Spam Classifier (Naive Bayes, no sklearn)
**Probability Meets Text Track** · Intermediate · ~60 min · 🟢 Colab only

## Scenario
Open the lesson narrative in `lab-steps.html` (same folder) for the full teaching text. This notebook is the **executable lab**: lesson notes as Markdown cells, runnable code as code cells, working against the dataset in this folder.

## You will learn
1. Tokenise SMS text and remove stopwords
2. Train multinomial Naive Bayes with Laplace smoothing from scratch
3. Evaluate on a seed=42 holdout with a confusion matrix
4. Rank spam-indicative tokens by log-likelihood ratio

## Datasets (this folder)
- `sms.tsv` — auto-download from `https://raw.githubusercontent.com/justmarkham/DAT8/master/data/sms.tsv`

## How to run on Google Colab
1. Click **Start Lab** — or open the hosted notebook directly: [Open in Colab](https://colab.research.google.com/github/matheshcp/ai_course_content/blob/main/course-01-foundations-python-math-data/labs/lab-06-naive-bayes-sms-spam/lab-06-naive-bayes-sms-spam.ipynb) — it opens under *your* Google account (Colab auto-saves a copy to your Drive; no per-student setup, no Drive API create).
2. Run **Cell 0 (bootstrap)** first — it pulls `dataset.zip` from the lab manifest into `/content/ml_lab` (falls back to public raw URLs, then local files).
3. **Runtime → Run all** (GPU not required for Course 1).
4. Work the **Exercises** cells before revealing **Solutions**.

> Direct-open flow: `Start Lab` → hosted URL → Cell 0 fetches `dataset.zip` from the manifest → `Runtime → Run all`.


### Setup (VLABS bootstrap)

Run the next cell (Cell 0) once. Fetch order: hosted `manifest.json` → `dataset.zip` extracted to `/content/ml_lab/<lab_id>` → per-file public raw URLs → local files next to this notebook. No-op when files already exist.


In [ ]:
# Cell 0 — VLABS bootstrap: run first. Works on Colab (direct-open URL) and locally.
import io, json, os, urllib.request, zipfile

LAB_ID = "lab-06-naive-bayes-sms-spam"
# Hosted manifest (matheshcp/ai_course_content, branch main).
MANIFEST_URL = "https://raw.githubusercontent.com/matheshcp/ai_course_content/main/course-01-foundations-python-math-data/bundles/lab-06-naive-bayes-sms-spam/manifest.json"
# Alternative: backend proxy to S3 — uncomment to use instead:
# MANIFEST_URL = f"https://api.vlabs.test/colab/{LAB_ID}/manifest"
ON_COLAB = os.path.isdir("/content")
DATA_DIR = f"/content/ml_lab/{LAB_ID}" if ON_COLAB else "."

def _fetch(url, timeout=30):
    with urllib.request.urlopen(url, timeout=timeout) as r:
        return r.read()

def _ensure_file(filename, url=None):
    """Local-first single-file fetch (also used by lesson load cells)."""
    for base in (DATA_DIR, "."):
        p = os.path.join(base, filename)
        if os.path.exists(p):
            print(f"found {p}")
            return p
    if not url:
        raise FileNotFoundError(
            f"{filename} missing: open via Start Lab (bundle) or add it next to the notebook")
    os.makedirs(DATA_DIR, exist_ok=True)
    dest = os.path.join(DATA_DIR, filename)
    print(f"downloading {filename} ...")
    urllib.request.urlretrieve(url, dest)
    print(f"saved {dest}")
    return dest

ensure = _ensure_file  # compat alias for lesson load cells

def vlabs_bootstrap():
    # 1) Hosted manifest -> dataset.zip -> DATA_DIR (direct-open path)
    try:
        m = json.loads(_fetch(MANIFEST_URL).decode("utf-8"))
        dz = m.get("dataset_zip")
        if dz:
            print(f"manifest ok: {MANIFEST_URL}")
            os.makedirs(DATA_DIR, exist_ok=True)
            zpath = os.path.join(DATA_DIR, "dataset.zip")
            urllib.request.urlretrieve(dz, zpath)
            with zipfile.ZipFile(zpath) as z:
                z.extractall(DATA_DIR)
            print(f"extracted dataset.zip -> {DATA_DIR}")
    except Exception as e:
        print(f"manifest skip ({e}); using file fallbacks")
    # 2) Per-file fallbacks (public raw URLs; local files are a no-op hit)
    _ensure_file("sms.tsv", "https://raw.githubusercontent.com/justmarkham/DAT8/master/data/sms.tsv")
    # 3) Work from the data dir on Colab so relative paths resolve
    if ON_COLAB and DATA_DIR != ".":
        os.chdir(DATA_DIR)
        print(f"cwd -> {DATA_DIR}")

vlabs_bootstrap()


## Probability Meets Text Track: Multinomial NB from Scratch

> **Scenario:** `sms.tsv` has ~5572 labeled SMS messages (ham/spam). Build a **multinomial Naive Bayes** classifier with Laplace smoothing using only the Python standard library — then evaluate on a 20% holdout with `seed=42`.
>
> **You will learn:** tokenisation, stopwords, word counts, log-priors, Bayes rule, train/test split, confusion matrix by hand.
> **Time:** ~60 minutes. **Level:** Intermediate. **Needs:** Python 3.8+ only. **Env:** 🟢 Colab only.

### Naive Bayes mental map

| Concept | Code shape |
|---|---|
| Prior P(label) | `count(label) / N` in log space |
| Likelihood P(word\|label) | `(count[word] + α) / (total + α·V)` |
| Score a message | `log prior + Σ log likelihood(token)` |
| Predict | `argmax` over the two labels |
| Holdout | `random.seed(42)` + `shuffle` + 80/20 split |

---

### 1. Load data (local first, Colab fallback)

In [ ]:
import csv, math, os, random, re
from collections import Counter

def load_sms():
    local = "sms.tsv"
    if not os.path.exists(local):
        import urllib.request
        urllib.request.urlretrieve(
            "https://raw.githubusercontent.com/justmarkham/DAT8/master/data/sms.tsv",
            local,
        )
    rows = []
    with open(local, newline="", encoding="utf-8") as f:
        for r in csv.reader(f, delimiter="\t"):
            if len(r) >= 2 and r[0] in ("ham", "spam"):
                rows.append((r[0], r[1]))
    return rows

msgs = load_sms()
print(len(msgs), "messages")
print(Counter(l for l, _ in msgs))
# 5572 messages  Counter({'ham': 4825, 'spam': 747})
print(msgs[0])


---

### 2. Tokenise

In [ ]:
STOP = set(
    "a an the and or but if in on at to for of is are was were be been "
    "i you he she it we they my your our their this that with as so not "
    "no do does did have has had will would can could".split()
)

def tokens(text):
    return [w for w in re.findall(r"[a-z0-9']+", text.lower())
            if w not in STOP and len(w) > 1]

print(tokens("FREE entry in 2 a weekly comp to win FA cup final tickets!"))
# ['free', 'entry', '2', 'weekly', 'comp', 'win', 'fa', 'cup', 'final', 'tickets']


---

### 3. Train / test split (seed = 42)

In [ ]:
random.seed(42)
shuffled = msgs[:]
random.shuffle(shuffled)
n_test = int(len(shuffled) * 0.2)
test, train = shuffled[:n_test], shuffled[n_test:]
print(len(train), len(test))   # 4458 1114
print("train labels:", Counter(l for l, _ in train))
# train labels: Counter({'ham': 3848, 'spam': 610})


---

### 4. Fit multinomial NB with Laplace (α = 1)

In [ ]:
ALPHA = 1.0
label_counts = Counter(l for l, _ in train)
N_train = len(train)
prior_log = {lab: math.log(label_counts[lab] / N_train) for lab in label_counts}
# ham ≈ -0.1471, spam ≈ -1.989

word_counts = {lab: Counter() for lab in label_counts}
vocab = set()
for lab, text in train:
    for w in tokens(text):
        word_counts[lab][w] += 1
        vocab.add(w)
V = len(vocab)
# vocab_size ≈ 7826

totals = {lab: sum(word_counts[lab].values()) + ALPHA * V for lab in label_counts}

def log_likelihood(lab, w):
    return math.log((word_counts[lab][w] + ALPHA) / totals[lab])

def classify(text):
    best, best_score = None, -math.inf
    for lab in label_counts:
        score = prior_log[lab]
        for w in tokens(text):
            if w in vocab:
                score += log_likelihood(lab, w)
            else:
                # OOV token: same α / total for every label → constant offset;
                # still include for correctness of the sum
                score += math.log(ALPHA / totals[lab])
        if score > best_score:
            best, best_score = lab, score
    return best

print(classify("Congratulations! You've won a free prize. Claim now!"))  # spam
print(classify("Hey, are we still on for lunch at noon?"))               # ham


---

### 5. Evaluate on holdout

In [ ]:
preds = [classify(t) for _, t in test]
gold  = [l for l, _ in test]
acc = sum(p == g for p, g in zip(preds, gold)) / len(gold)
print(f"accuracy: {acc:.4f}")  # 0.9803
print("correct:", sum(p == g for p, g in zip(preds, gold)), "/", len(gold))  # 1092 / 1114

conf = Counter(zip(gold, preds))
print("ham->ham", conf[("ham", "ham")], " ham->spam", conf[("ham", "spam")])
print("spam->ham", conf[("spam", "ham")], " spam->spam", conf[("spam", "spam")])
# ham->ham 962  ham->spam 15
# spam->ham 7   spam->spam 130


Confusion matrix layout (rows = true, cols = predicted):

|  | pred ham | pred spam |
|---|---|---|
| **true ham** | 962 | 15 |
| **true spam** | 7 | 130 |

---

### 6. Top spam-indicative tokens (log lift)

In [ ]:
spam_lift = []
for w in vocab:
    ps = (word_counts["spam"][w] + ALPHA) / totals["spam"]
    ph = (word_counts["ham"][w] + ALPHA) / totals["ham"]
    spam_lift.append((w, math.log(ps / ph),
                      word_counts["spam"][w], word_counts["ham"][w]))
spam_lift.sort(key=lambda x: -x[1])
top = [(w, round(lift, 3), s, h) for w, lift, s, h in spam_lift if s + h >= 10][:10]
for row in top:
    print(row)
# claim 5.343 (90 spam / 0 ham in train)
# prize 5.109
# won 4.959
# 150p 4.875
# tone 4.724
# ...


---

## Exercises (do these!)

### Exercise 1 — Top-10 spam-indicative tokens
Using training counts only, rank tokens by `log P(w|spam) − log P(w|ham)` (support ≥ 10). Print the top 10.
*Expected (approx): claim, prize, won, 150p, tone, 18, www, guaranteed, 1000, 500.*

<details>
<summary>Hint</summary>

Same computation as Section 6; filter `s + h >= 10` before sorting.
</details>

### Exercise 2 — Accuracy on 20% holdout (seed 42)
Report accuracy to 4 d.p. and the 4 confusion counts.
*Expected: accuracy ≈ 0.9803 · TP_spam=130, FN_spam=7, FP_spam=15, TN=962.*

<details>
<summary>Hint</summary>

`random.seed(42); random.shuffle(...)` **before** slicing — order of operations matters.
</details>

### Exercise 3 — Why “free” alone misclassifies marketing ham
How many holdout messages contain `"free"`? How many of those are **ham**? Give one example of a ham message with “free” that the bag-of-words model might over-weight.
*Expected: 48 test messages contain “free”; 14 are ham (e.g. “When you get free, call me”). Naive Bayes multiplies independent token evidence — a rare spammy co-occurrence pattern can tip borderline ham over the decision boundary.*

<details>
<summary>Hint</summary>

```python
free_test = [(t, g) for (g, t) in test if "free" in t.lower()]
print(len(free_test), sum(1 for _, g in free_test if g == "ham"))
```

Single tokens cannot capture context; that’s the “naive” independence assumption.
</details>

---

## Solutions

In [ ]:
# --- Solution 1 ---
lift = []
for w in vocab:
    ps = (word_counts["spam"][w] + ALPHA) / totals["spam"]
    ph = (word_counts["ham"][w] + ALPHA) / totals["ham"]
    s = word_counts["spam"][w]; h = word_counts["ham"][w]
    if s + h >= 10:
        lift.append((w, math.log(ps/ph), s, h))
lift.sort(key=lambda x: -x[1])
print([w for w, *_ in lift[:10]])
# ['claim', 'prize', 'won', '150p', 'tone', '18', 'www', 'guaranteed', '1000', '500']

# --- Solution 2 ---
preds = [classify(t) for _, t in test]
gold  = [l for l, _ in test]
acc = sum(p == g for p, g in zip(preds, gold)) / len(gold)
print(f"{acc:.4f}")  # 0.9803
print(Counter(zip(gold, preds)))
# 962 ham->ham, 15 ham->spam, 7 spam->ham, 130 spam->spam

# --- Solution 3 ---
free_test = [(t, g) for (g, t) in test if "free" in t.lower()]
n_free = len(free_test)
n_ham = sum(1 for _, g in free_test if g == "ham")
example = next(t for t, g in free_test if g == "ham")
print(n_free, n_ham)
print(example[:80])
# 48 14
# e.g. "When you get free, call me" / "Now am free call me pa."


### What to learn next
- Add bigrams (`"free entry"`) — reduces “free” false positives.
- Compare to `sklearn.naive_bayes.MultinomialNB` (should land ≈ same accuracy).
- TF-IDF weights; threshold tuning for precision vs recall on spam.
- Cheat sheet: tokenize → count → log-priors + log-likelihoods → argmax → confusion matrix.

*Files in this folder: `sms.tsv`. Paste any block into Python/Jupyter and run top-to-bottom.*

---

**Done with Colab?** Download the notebook (**File → Download .ipynb**) to keep outputs, or **File → Save a copy in Drive**. Re-upload datasets after a runtime recycle.
